In [12]:
import numpy as np
import pickle
import random
from collections import defaultdict

class SupplyChainAgent:
    def __init__(self):
        self.action_size = 5
        self.actions = {
            0: "do_nothing",
            1: "reorder_stock",
            2: "switch_supplier",
            3: "reroute_shipment",
            4: "emergency_restock"
        }

        self.epsilon       = 1.0
        self.epsilon_min   = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.05   # increased from 0.001 — was too slow to learn
        self.gamma         = 0.95

        self.q_table = defaultdict(lambda: np.zeros(self.action_size))

    def get_state_key(self, state):
        demand = state["predicted_demand"]
        if demand < 400000:
            demand_bucket = 0
        elif demand < 700000:
            demand_bucket = 1
        else:
            demand_bucket = 2

        # Fix: recalibrated thresholds for large demand values
        inventory = state["current_inventory"]
        ratio = inventory / max(demand, 1)

        if ratio < 0.02:
            inv_ratio = 0   # critical  (< 2% of demand)
        elif ratio < 0.10:
            inv_ratio = 1   # low       (2-10%)
        elif ratio < 0.30:
            inv_ratio = 2   # medium    (10-30%)
        elif ratio < 0.60:
            inv_ratio = 3   # adequate  (30-60%)
        else:
            inv_ratio = 4   # healthy   (> 60%)

        risk_bucket       = int(state["supplier_risk_score"] * 5)
        disruption_bucket = int(state["disruption_signal"] * 5)

        days = state["days_to_stockout"]
        if days <= 3:
            days_bucket = 0
        elif days <= 7:
            days_bucket = 1
        elif days <= 14:
            days_bucket = 2
        else:
            days_bucket = 3

        return (demand_bucket, inv_ratio, risk_bucket, disruption_bucket, days_bucket)

    def choose_action(self, state, training=True):
        # Exploration only during training
        if training and random.random() < self.epsilon:
            return random.randint(0, self.action_size - 1)

        key = self.get_state_key(state)
        return int(np.argmax(self.q_table[key]))

    def learn(self, state, action, reward, next_state):
        key      = self.get_state_key(state)
        next_key = self.get_state_key(next_state)

        current_q = self.q_table[key][action]
        target_q  = reward + self.gamma * np.max(self.q_table[next_key])

        self.q_table[key][action] += self.learning_rate * (target_q - current_q)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def save(self, path='supply_chain_agent.pkl'):
        with open(path, 'wb') as f:
            pickle.dump(dict(self.q_table), f)
        print(f"Agent saved to {path}")

    def load(self, path='supply_chain_agent.pkl'):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        self.q_table = defaultdict(lambda: np.zeros(self.action_size), data)
        self.epsilon = self.epsilon_min
        print(f"Agent loaded from {path}")


def calculate_reward(action, outcome):
    reward = 0

    if outcome["stockout_avoided"]:
        reward += 80

    if outcome["delay_avoided"]:
        reward += 40

    if outcome["cost_saved"] > 0:
        reward += outcome["cost_saved"] / 500   # stronger scaling

    if outcome["stockout_occurred"]:
        reward -= 300   # BIG penalty

    if outcome["unnecessary_action"]:
        reward -= 50

    if outcome["cost_increased"] > 0:
        reward -= outcome["cost_increased"] / 500

    return reward


def simulate_outcome(state, action):
    outcome = {
        "stockout_avoided":   False,
        "stockout_occurred":  False,
        "cost_saved":         0,
        "delay_avoided":      False,
        "unnecessary_action": False,
        "cost_increased":     0
    }

    # -------------------------------
    # DERIVED CONDITIONS
    # -------------------------------
    demand = max(state["predicted_demand"], 1)
    inventory = state["current_inventory"]
    ratio = inventory / demand

    low_inventory   = ratio < 0.10
    critical_inv    = ratio < 0.02

    high_risk       = state["supplier_risk_score"] > 0.7
    extreme_risk    = state["supplier_risk_score"] > 0.85

    high_disruption = state["disruption_signal"] > 0.6
    extreme_disrupt = state["disruption_signal"] > 0.85

    critical_days   = state["days_to_stockout"] <= 7
    urgent_days     = state["days_to_stockout"] <= 3

    # -------------------------------
    # ACTION LOGIC
    # -------------------------------

    # 🚫 DO NOTHING
    if action == 0:
        if critical_inv or urgent_days:
            outcome["stockout_occurred"] = True
            outcome["cost_increased"] = 600   # severe penalty

        elif low_inventory or critical_days:
            outcome["stockout_occurred"] = True
            outcome["cost_increased"] = 400

        elif high_risk or high_disruption:
            outcome["cost_increased"] = 250   # passive loss

        else:
            outcome["cost_saved"] = 50  # saving operational cost


    # 📦 REORDER STOCK
    elif action == 1:
        if low_inventory or critical_days:
            outcome["stockout_avoided"] = True
            outcome["cost_saved"]       = 600

            if high_risk:
                outcome["delay_avoided"] = True
                outcome["cost_saved"]   += 200

        else:
            outcome["unnecessary_action"] = True
            outcome["cost_increased"]     = 250


    # 🔁 SWITCH SUPPLIER
    elif action == 2:
        if extreme_risk:
            outcome["delay_avoided"] = True
            outcome["cost_saved"]    = 500

            if low_inventory:
                outcome["stockout_avoided"] = True
                outcome["cost_saved"]      += 300

        elif high_risk:
            outcome["delay_avoided"] = True
            outcome["cost_saved"]    = 300

        else:
            outcome["unnecessary_action"] = True
            outcome["cost_increased"]     = 200


    # 🚚 REROUTE SHIPMENT
    elif action == 3:
        if extreme_disrupt:
            outcome["delay_avoided"] = True
            outcome["cost_saved"]    = 400

            if low_inventory:
                outcome["stockout_avoided"] = True
                outcome["cost_saved"]      += 250

        elif high_disruption:
            outcome["delay_avoided"] = True
            outcome["cost_saved"]    = 250

        else:
            outcome["unnecessary_action"] = True
            outcome["cost_increased"]     = 150


    # 🚨 EMERGENCY RESTOCK
    elif action == 4:
        if urgent_days or critical_inv:
            outcome["stockout_avoided"] = True
            outcome["cost_saved"]       = 700   # highest impact

        elif critical_days:
            outcome["stockout_avoided"] = True
            outcome["cost_saved"]       = 400

        else:
            outcome["unnecessary_action"] = True
            outcome["cost_increased"]     = 600  # expensive mistake


    return outcome

def build_training_states(forecast_df, n_episodes=2000):
    states = []
    predicted_sales_list = forecast_df['predicted_sales'].tolist()

    for _ in range(n_episodes):
        base_demand  = random.choice(predicted_sales_list)
        demand_noise = base_demand * random.uniform(0.85, 1.15)

        scenario = random.choice(['critical', 'low', 'medium', 'adequate', 'healthy'])

        if scenario == 'critical':
            # Under 2% of demand — emergency territory
            inventory  = random.uniform(0, demand_noise * 0.02)
            days       = random.randint(1, 3)
            risk       = random.uniform(0.6, 1.0)
            disruption = random.uniform(0.5, 1.0)

        elif scenario == 'low':
            # 2-10% of demand — reorder territory
            inventory  = random.uniform(demand_noise * 0.02, demand_noise * 0.10)
            days       = random.randint(4, 10)
            risk       = random.uniform(0.3, 0.8)
            disruption = random.uniform(0.2, 0.7)

        elif scenario == 'medium':
            # 10-30% — watch territory, switch supplier or reroute if risk high
            inventory  = random.uniform(demand_noise * 0.10, demand_noise * 0.30)
            days       = random.randint(10, 18)
            risk       = random.uniform(0.6, 1.0)   # high risk to trigger switch
            disruption = random.uniform(0.6, 1.0)   # high disruption to trigger reroute

        elif scenario == 'adequate':
            # 30-60% — mostly fine, occasional risk
            inventory  = random.uniform(demand_noise * 0.30, demand_noise * 0.60)
            days       = random.randint(15, 25)
            risk       = random.uniform(0.0, 0.5)
            disruption = random.uniform(0.0, 0.5)

        else:  # healthy
            # Over 60% — do nothing territory
            inventory  = random.uniform(demand_noise * 0.60, demand_noise * 1.20)
            days       = random.randint(20, 30)
            risk       = random.uniform(0.0, 0.3)
            disruption = random.uniform(0.0, 0.3)

        states.append({
            "predicted_demand":    demand_noise,
            "current_inventory":   float(inventory),
            "supplier_risk_score": float(risk),
            "disruption_signal":   float(disruption),
            "days_to_stockout":    float(days)
        })

    return states


def train_agent(forecast_df, episodes=2000):
    agent  = SupplyChainAgent()
    states = build_training_states(forecast_df, n_episodes=episodes)

    total_rewards = []
    action_counts = defaultdict(int)

    print(f"Training agent for {episodes} episodes...")
    print("-" * 50)

    for episode, state in enumerate(states):
        action = agent.choose_action(state, training=True)
        outcome = simulate_outcome(state, action)
        reward  = calculate_reward(action, outcome)

        next_state = {
            "predicted_demand":    state["predicted_demand"],
            "current_inventory":   max(0.0, state["current_inventory"]
                                    - state["predicted_demand"] * 0.1
                                    + (500.0 if action in [1, 4] else 0.0)),
            "supplier_risk_score": max(0.0, state["supplier_risk_score"]
                                    - (0.2 if action == 2 else 0.0)),
            "disruption_signal":   max(0.0, state["disruption_signal"]
                                    - (0.1 if action == 3 else 0.0)),
            "days_to_stockout":    max(0.0, state["days_to_stockout"] - 1.0
                                    + (7.0 if action in [1, 4] else 0.0))
        }

        agent.learn(state, action, reward, next_state)
        total_rewards.append(reward)
        action_counts[agent.actions[action]] += 1

        if (episode + 1) % 400 == 0:
            avg_reward = np.mean(total_rewards[-400:])
            print(f"  Episode {episode+1:>4} | Avg reward: {avg_reward:>7.2f} | Epsilon: {agent.epsilon:.3f}")

    print("-" * 50)
    print(f"Training complete.")
    print(f"Q-table entries learned: {len(agent.q_table)}")
    print(f"\nAction distribution:")
    for action_name, count in sorted(action_counts.items()):
        print(f"  {action_name:<22} {count:>5} times ({count/episodes*100:.1f}%)")

    agent.save('supply_chain_agent.pkl')
    return agent


def get_recommendation(state, agent=None):
    if agent is None:
        agent = SupplyChainAgent()
        agent.load('supply_chain_agent.pkl')

    # Always disable exploration during inference
    agent.epsilon = 0.0

    ratio = state["current_inventory"] / max(state["predicted_demand"], 1)

    # -------------------------------
    # RULE-BASED SAFETY LAYER
    # -------------------------------
    if ratio < 0.02 or state["days_to_stockout"] <= 3:
        action_name = "emergency_restock"
        confidence = 95.0   # HIGH confidence for rule-based decisions

    elif state["supplier_risk_score"] > 0.8:
        action_name = "switch_supplier"
        confidence = 90.0

    elif state["disruption_signal"] > 0.8:
        action_name = "reroute_shipment"
        confidence = 90.0

    else:
        action_id = agent.choose_action(state, training=False)
        action_name = agent.actions[action_id]

    # -------------------------------
    # EXPLANATION ENGINE
    # -------------------------------
    reasons = []

    if state["days_to_stockout"] <= 3:
        reasons.append("CRITICAL: under 3 days of stock remaining")
    elif state["days_to_stockout"] <= 7:
        reasons.append(f"urgent: only {state['days_to_stockout']:.0f} days left")

    if ratio < 0.02:
        reasons.append("inventory critically below 2% of demand")
    elif ratio < 0.10:
        reasons.append("inventory below 10% of demand")
    elif ratio < 0.30:
        reasons.append("inventory below 30% of demand")

    if state["supplier_risk_score"] > 0.7:
        reasons.append(f"supplier risk high ({state['supplier_risk_score']:.2f})")

    if state["disruption_signal"] > 0.6:
        reasons.append(f"disruption signal high ({state['disruption_signal']:.2f})")

    if not reasons:
        reasons.append("all metrics within normal range")

    explanation = f"Action '{action_name.upper()}' chosen because: " + " + ".join(reasons)

    # -------------------------------
    # CONFIDENCE SCORE (from Q-table)
    # -------------------------------
    try:
        key = agent.get_state_key(state)
        q_values = agent.q_table[key]
        confidence = float(np.max(q_values))

        # Normalize confidence to 0–100
        confidence = max(0, min(100, (confidence + 100) / 2))
    except:
        confidence = 0.0

    # -------------------------------
    # PRINT OUTPUT
    # -------------------------------
    print("\n" + "=" * 55)
    print("  AI SUPPLY CHAIN DECISION ENGINE")
    print("=" * 55)
    print(f"  Action:      {action_name.upper()}")
    print(f"  Confidence:  {confidence:.2f}")
    print(f"\n  Explanation:")
    print(f"  - " + "\n  - ".join(reasons))

    print(f"\n  State Summary:")
    for k, v in state.items():
        if isinstance(v, float):
            print(f"    {k:<25} {v:.2f}")
        else:
            print(f"    {k:<25} {v}")

    print("=" * 55)

    return {
        "action": action_name,
        "confidence": confidence,
        "explanation": explanation,
        "state": state
    }

In [ ]:
import os
import json
import requests
from datetime import timedelta

# ---------- Layer 1: Data Collection ----------

import os
from layer1 import build_layer1_dataset

# Example usage: replace NEWSDATA_API_KEY and WEATHERAPI_KEY with your real keys when available.
# data_bundle = build_layer1_dataset(
#     news_api_key=os.getenv('NEWSDATA_API_KEY'),
#     weather_api_key=os.getenv('WEATHERAPI_KEY')
# )
# data_bundle['layer1_df'].head()


In [13]:
# ── Run everything (Jupyter-compatible) ───────────────────────────
import pandas as pd
import xgboost as xgb
import pickle
import numpy as np

# ── Load saved models ──────────────────────────────────────────────
with open('demand_forecast_model.pkl', 'rb') as f:
    model_tuned = pickle.load(f)

with open('demand_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# ── Load and prep data ─────────────────────────────────────────────
store_sales = pd.read_csv('store_sale.csv')
store_sales = store_sales.drop(['store', 'item'], axis=1)
store_sales['date'] = pd.to_datetime(store_sales['date'])
store_sales['date'] = store_sales['date'].dt.to_period('M')
monthly_sales = store_sales.groupby('date').sum().reset_index()
monthly_sales['date'] = monthly_sales['date'].dt.to_timestamp()
monthly_sales['sales_diff'] = monthly_sales['sales'].diff()
monthly_sales = monthly_sales.dropna().reset_index(drop=True)

# ── Lag feature builder ────────────────────────────────────────────
def build_lag_features(series, n_lags=12):
    df = pd.DataFrame(series, columns=['sales_diff'])
    for i in range(1, n_lags + 1):
        df[f'month_{i}'] = df['sales_diff'].shift(i)
    df = df.dropna().reset_index(drop=True)
    return df

# ── Rolling N-month forecast ───────────────────────────────────────
def predict_n_months(n=3):
    temp_monthly = monthly_sales.copy()
    predictions  = []

    for i in range(n):
        temp_supervised = build_lag_features(temp_monthly['sales_diff'])
        latest_features = temp_supervised.iloc[-1]
        input_features  = latest_features[[f'month_{j}' for j in range(1, 13)]].values

        full_input = np.concatenate([[0], input_features]).reshape(1, -1)

        # Fix: named DataFrame to silence scaler warning
        feature_names  = ['sales_diff'] + [f'month_{i}' for i in range(1, 13)]
        full_input_df  = pd.DataFrame(full_input, columns=feature_names)
        full_input_scaled = scaler.transform(full_input_df)

        X_input       = full_input_scaled[:, 1:]
        dmatrix_input = xgb.DMatrix(X_input)
        scaled_pred   = model_tuned.predict(dmatrix_input)

        pred_row      = np.concatenate([scaled_pred.reshape(-1, 1), X_input], axis=1)
        pred_unscaled = scaler.inverse_transform(pred_row)
        predicted_diff = pred_unscaled[0][0]

        last_sales      = temp_monthly['sales'].iloc[-1]
        predicted_sales = predicted_diff + last_sales
        next_date       = temp_monthly['date'].iloc[-1] + pd.DateOffset(months=1)

        predictions.append({
            'month':           next_date.strftime('%B %Y'),
            'predicted_sales': round(predicted_sales),
            'sales_diff':      round(predicted_diff)
        })

        new_row = pd.DataFrame({
            'date':       [next_date],
            'sales':      [predicted_sales],
            'sales_diff': [predicted_diff]
        })
        temp_monthly = pd.concat([temp_monthly, new_row], ignore_index=True)

    return pd.DataFrame(predictions)

# ── Step 1: Get forecast ───────────────────────────────────────────
forecast_df = predict_n_months(n=3)
print("Forecast used for training states:")
print(forecast_df.to_string(index=False))
print()

# ── Step 2: Train the RL agent ─────────────────────────────────────
agent = train_agent(forecast_df, episodes=2000)

# ── Step 3: Test with a realistic crisis state ─────────────────────
live_state = {
    "predicted_demand":    float(forecast_df['predicted_sales'].iloc[0]),
    "current_inventory":   800.0,
    "supplier_risk_score": 0.82,
    "disruption_signal":   0.71,
    "days_to_stockout":    5.0
}
agent.epsilon = 0.0
result = get_recommendation(live_state, agent)

# ── Step 4: Test with a healthy state ─────────────────────────────
healthy_state = {
    "predicted_demand":    float(forecast_df['predicted_sales'].iloc[0]),
    "current_inventory":   50000.0,
    "supplier_risk_score": 0.15,
    "disruption_signal":   0.10,
    "days_to_stockout":    25.0
}

result_healthy = get_recommendation(healthy_state, agent)

Forecast used for training states:
        month  predicted_sales  sales_diff
 January 2018           472277     -222893
February 2018           388026      -84251
   March 2018           363468      -24558

Training agent for 2000 episodes...
--------------------------------------------------
  Episode  400 | Avg reward:  -11.42 | Epsilon: 0.135
  Episode  800 | Avg reward:   34.96 | Epsilon: 0.018
  Episode 1200 | Avg reward:   40.00 | Epsilon: 0.010
  Episode 1600 | Avg reward:   44.15 | Epsilon: 0.010
  Episode 2000 | Avg reward:   41.14 | Epsilon: 0.010
--------------------------------------------------
Training complete.
Q-table entries learned: 233

Action distribution:
  do_nothing               804 times (40.2%)
  emergency_restock        179 times (8.9%)
  reorder_stock            437 times (21.9%)
  reroute_shipment         335 times (16.8%)
  switch_supplier          245 times (12.2%)
Agent saved to supply_chain_agent.pkl

  AI SUPPLY CHAIN DECISION ENGINE
  Action:      EM

In [14]:
# Test switch_supplier
high_risk_state = {
    "predicted_demand":    float(forecast_df['predicted_sales'].iloc[0]),
    "current_inventory":   30000.0,
    "supplier_risk_score": 0.90,
    "disruption_signal":   0.30,
    "days_to_stockout":    20.0
}
get_recommendation(high_risk_state, agent)

# Test reroute_shipment
disruption_state = {
    "predicted_demand":    float(forecast_df['predicted_sales'].iloc[0]),
    "current_inventory":   25000.0,
    "supplier_risk_score": 0.25,
    "disruption_signal":   0.85,
    "days_to_stockout":    18.0
}
get_recommendation(disruption_state, agent)

# Test emergency_restock
critical_state = {
    "predicted_demand":    float(forecast_df['predicted_sales'].iloc[0]),
    "current_inventory":   200.0,
    "supplier_risk_score": 0.80,
    "disruption_signal":   0.75,
    "days_to_stockout":    2.0
}
get_recommendation(critical_state, agent)


  AI SUPPLY CHAIN DECISION ENGINE
  Action:      SWITCH_SUPPLIER
  Confidence:  50.00

  Explanation:
  - inventory below 10% of demand
  - supplier risk high (0.90)

  State Summary:
    predicted_demand          472277.00
    current_inventory         30000.00
    supplier_risk_score       0.90
    disruption_signal         0.30
    days_to_stockout          20.00

  AI SUPPLY CHAIN DECISION ENGINE
  Action:      REROUTE_SHIPMENT
  Confidence:  50.00

  Explanation:
  - inventory below 10% of demand
  - disruption signal high (0.85)

  State Summary:
    predicted_demand          472277.00
    current_inventory         25000.00
    supplier_risk_score       0.25
    disruption_signal         0.85
    days_to_stockout          18.00

  AI SUPPLY CHAIN DECISION ENGINE
  Action:      EMERGENCY_RESTOCK
  Confidence:  100.00

  Explanation:
  - CRITICAL: under 3 days of stock remaining
  - inventory critically below 2% of demand
  - supplier risk high (0.80)
  - disruption signal high (0

{'action': 'emergency_restock',
 'confidence': 100,
 'explanation': "Action 'EMERGENCY_RESTOCK' chosen because: CRITICAL: under 3 days of stock remaining + inventory critically below 2% of demand + supplier risk high (0.80) + disruption signal high (0.75)",
 'state': {'predicted_demand': 472277.0,
  'current_inventory': 200.0,
  'supplier_risk_score': 0.8,
  'disruption_signal': 0.75,
  'days_to_stockout': 2.0}}